In [ ]:
import pandas as pd
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
import warnings
warnings.filterwarnings("ignore")
import numpy as np
from sklearn.impute import SimpleImputer
import seaborn as sns
import streamlit as st
import matplotlib.pyplot as plt
import xgboost as xgb
from snowflake.snowpark.context import get_active_session
session = get_active_session()

from snowflake.snowpark import DataFrame
from snowflake.snowpark.functions import col, to_timestamp, min, max, month, dayofweek, dayofyear, avg, date_add, sql_expr
from snowflake.snowpark.types import IntegerType
from snowflake.snowpark import Window

In [ ]:
df = pd.read_csv("boat_data_.csv")

df.head(10)

In [ ]:
USE SNOW_ML_DB.SNOW_ML_SCHEMA;
USE WAREHOUSE SNOW_ML_PIPELINE;

Add all PDF files to the boats_stage to run the next cell

In [ ]:
LIST @boats_stage;

In [ ]:
-- Extracts data, text and layout elements from documents that are stored in a STAGE

-- SELECT AI_PARSE_DOCUMENT(
--     TO_FILE('@boats_stage', 'catamaran_000.pdf'),
--         {'mode': 'LAYOUT', 'page_split':true}) AS pdf_output_data

In [ ]:
# {
#   "metadata": {
#     "pageCount": 2
#   },
#   "pages": [
#     {
#       "content": "![img-0.jpeg](img-0.jpeg)\n\nVessel ID: LAG450F-2016-CHS-1027\nTarget Type: Sailing Catamaran
#       \nHull Count: 2\nHull Type: Catamaran\nPropulsion: Sail + Twin Inboard Diesel\nLength Overall (ft): 45.83\nBeam (ft): 25.75
#       \nDraft (ft): 4.27\nDisplacement (kg): 15500\nHull Material: Fiberglass (GRP)\nEngine Count: 2\nEngine Type: Inboard\nFuel Type: Diesel
#       \nSail Area (sq ft): 1600\nMast Count: 1\nRig Type: Fractional Sloop (Marconi)\nCabin Count: 4\nBerth Count: 8\nYear Built: 2016
#       \nPrice (USD): 579000\nLocation Country: United States\nBuilder: Lagoon\nModel: 450 F",
#       "index": 0
#     },
#     {
#       "content": "Condition: Used - Excellent\nEngine Hours: 2150\nMax Speed (knots): 9.2\nCruising Speed (knots): 7.5\nRange (nm): 800
#       \nFuel Capacity (L): 1000\nWater Capacity (L): 700\nGenerator Onboard: True\nAir Conditioning: True\nFlybridge: True\nHelm Count: 1
#       \nBeam-to-Length Ratio: 0.56\nBrand Popularity Score: 9.1\nElectronics Score: 9.0\nSafety Gear Score: 8.8\nPhotos Count: 42\n
#       Has Trailer: False\nCreated At: 2025-11-14T15:22:00Z\nListing Description:\nWell-kept 2016 Lagoon 450 F flybridge catamaran in excellent condition, privately owned (never changed).
#       \n\nHighlights and equipment:\n\n- Twin Yanmar 4JH57 57 HP inboard diesels (approx. 2,150 hours), serviced on schedule
#       \n- Onan 11 kW diesel generator (approx. 1,200 hours) with sound shield\n- 1,200 W solar array with MPPT controllers; 600 Ah lithium house bank (LiFePO4) plus dedicated 2000
#       \n- Air conditioning throughout (approx. 48,000 BTU total) with digital thermostats\n- Standing rigging inspected 2022; new running rigging 2023 (main halyard, jib sheets, traveler lines)
#       \n- Fully battened mainsail with lazy bag; furling genoa; Code 0 on furler for light-air performance\n- Electric primary winch at helm; all lines led aft for safe, easy shorthanded sailing
#       \n- Raymarine Axiom MFDs (flybridge and nav station), Quantum radar, AIS transceiver, autopilot with\n- Full safety inventory: offshore liferaft (2023 service), EPIRB, jacklines, MOB recovery gear, and c
#       \n- 10' RIB tender with 15 HP outboard on stainless davits; electric davit winch\n- $60 \\mathrm{~L} / \\mathrm{hr}$ watermaker (recent membranes, 2024); electric freshwater heads\
#       n- Ground tackle: 33 kg Rocna primary with 100 m chain, secondary Fortress kedge; windlass with w\n- Recent bottom paint (2024), saildrive seals (2023), heat exchangers and raw-water pumps service
#       \n- Interior: 4 queen cabins each with ensuite; LED lighting; large U-shaped galley up with 3-burner st\n- Entertainment: Fusion stereo with cockpit and saloon zones, smart TV, Wi-Fi router/booster
#       \n- Cockpit sunshades, helm bimini, and foredeck lounge cushions; cockpit fridge\n\nPerformance and range:\n\n- Comfortable cruising at $7.0-7.5$ knots under power; typical sailing speeds $7-10$ knots with double
#       \n- Approximate motoring range of 800 nm at conservative RPMs with 1,000 L diesel and proper sea\n\nThis Lagoon 450 F presents as a turn-key, well-specified flybridge catamaran ideal for family cruisin",
#       "index": 1
#     }
#   ]
# }

In [ ]:
-- WITH scan_doc AS (
--     SELECT AI_PARSE_DOCUMENT(
--     TO_FILE('@boats_stage', 'catamaran_000.pdf'),
--         {'mode': 'LAYOUT', 'page_filter': [{'start': 0, 'end': 2}]}
--     ) AS pdf_parser
-- )
-- SELECT 
--     AI_COMPLETE('claude-3-5-sonnet',
--                 'You are a text extractor. Extract the following data points: vessel_id, target_type, hull_count and hull_type from the following document:' || pdf_parser::VARCHAR
--     ) AS pdf_data
-- FROM scan_doc;

"Here are the extracted data points:\n\n* 
`vessel_id`: LAG450F-2016-CHS-1027\n* 
`target_type`: Sailing Catamaran\n* 
`hull_count`: 2\n* 
`hull_type`: Catamaran\n
\nLet me know if you need any further assistance!"


In [ ]:
# pdf_list = []

# for x in range(2):
#     sql_query = f"""
#                 WITH t1 AS (
#                     SELECT AI_PARSE_DOCUMENT(
#                     TO_FILE('@boats_stage', 'catamaran_00{x}.pdf'),
#                         {{'mode': 'LAYOUT', 'page_filter': [{{'start': 0, 'end': 2}}]}}
#                     ) AS parser
#                 )
#                 SELECT 
#                     AI_COMPLETE('claude-3-5-sonnet',
#                                 'You are a text extractor. 
#                                  Extract the following data points 
#                                  vessel_id, target_type, hull_count and hull_type 
#                                  from the following document:' || parser::VARCHAR
#                     ) AS hey
#                 FROM t1;
#                 """
    
#     sql_session = session.sql(sql_query)

#     query_output = sql_session.collect()
#     pdf_list.append(query_output)

    
# for pdf_data in pdf_list:
#     print(pdf_data, "\n")

Data extracted from PDF files

In [ ]:
catamaran_df = pd.read_csv("catamaran_extracted.csv")

power_catamaran_df = pd.read_csv("power_catamaran_extracted.csv")

monohull_df = pd.read_csv("monohull_extracted.csv")

In [ ]:
df = pd.concat(
               [df, catamaran_df, power_catamaran_df, monohull_df], 
                ignore_index = True
               ) 

In [ ]:
print(len(df))

Eploratory Data Analysis 📊 

In [ ]:
df.shape

df.head(10000)

In [ ]:
df.describe()

In [ ]:
print(df.columns)

print(df.dtypes)

In [ ]:
df = df[[   
       'target_type', 'vessel_id','hull_count', 'hull_type', 'beam_to_length_ratio', 
       'length_overall_ft', 'beam_ft', 'draft_ft',   
       'engine_count', 'engine_type', 'fuel_type', 'propulsion', 
        'mast_count', 'rig_type', 'year_built', 
        'feature_premium_cruising_score'
     ]].copy()

In [ ]:
# applying title case to each column
df.columns = df.columns.str.title()

print(df.columns)

In [ ]:
# finding what are the missing values and the percentages
print(df.isna().sum())

In [ ]:
hull_num = df['Hull_Count'].value_counts()

st.bar_chart(hull_num)

Data Cleaning 🧹

In [ ]:
# converting string values to integer

print("Unique values on the column", df["Hull_Count"].unique())
print("NaN values:", df["Hull_Count"].isna().sum())

mylist = []

print()
for hull_c in df["Hull_Count"]:
    try:
        int(hull_c)
        
    except Exception as error:
        print("Exception:", error)
        w_index = df[df["Hull_Count"] == hull_c].index

        print(w_index)
        mylist.extend(w_index)

df = df.drop(index=mylist)

In [ ]:
# changing data types

df["Hull_Count"] = df["Hull_Count"].astype("int")

df["Vessel_Id"] = df["Vessel_Id"].astype('string')

df["Hull_Type"] = df["Hull_Type"].astype('string')

df["Target_Type"] = df["Target_Type"].astype('string').str.upper()

df["Hull_Type"] = df["Hull_Type"].astype('string').str.upper()

df["Propulsion"] = df["Propulsion"].astype('string').str.upper()

In [ ]:
# replacing string with integer values
# two_thousand => 2000

mapping = {
    "two thousand": 2000
}

df["Year_Built"] = (
    df["Year_Built"]
    .astype("string")          
    .str.lower().str.strip()   
    .replace(mapping)          
    .astype("int64")           
)

In [ ]:
# total amount of missing values
nulls = df['Beam_To_Length_Ratio'].isna().sum()

# total amount of not missing values
not_nulls = df['Beam_To_Length_Ratio'].notna().sum()

counting = [nulls, not_nulls]
labeling = ['NaN', 'Not NaN']

# make the figure and pie smaller
fig, ax = plt.subplots(figsize=(4, 4)) 
ax.pie(
    counting,
    labels=labeling,
    autopct='%1.1f%%',
    startangle=90,
    radius=0.7        
)

print('Beam_To_Length_Ratio')
ax.axis('equal')       
plt.tight_layout()
plt.show()


In [ ]:
df['Beam_To_Length_Ratio']

In [ ]:
# statistical imputation 
# replacing missing values with the mean value

imp = SimpleImputer(strategy='mean')

imp.fit(df[['Beam_To_Length_Ratio']])

imp_transformation = imp.transform(df[['Beam_To_Length_Ratio']])

df['Beam_To_Length_Ratio'] = imp_transformation.round(3)

df['Beam_To_Length_Ratio']

In [ ]:
# total percentage of missing values
nulls = df['Beam_To_Length_Ratio'].isna().sum()

# total percentage of not missing values
not_nulls = df['Beam_To_Length_Ratio'].notna().sum()

counting = [nulls, not_nulls]
labeling = ['NaN', 'Not NaN']


fig, ax = plt.subplots(figsize=(4, 4))  

ax.pie(
    counting,
    labels=labeling,
    autopct='%1.1f%%',
    startangle=90,
    radius=0.7        
)

ax.axis('equal')       
plt.tight_layout()
plt.show()

In [ ]:
df[["Length_Overall_Ft", "Beam_Ft", "Draft_Ft"]]

In [ ]:
def cleaning_col(column_name):
    
    df[column_name] = df[column_name].astype("string")
    
    # Count how many values in the column contain a comma
    comma_count = df[column_name].str.contains(",").sum()
    
    # Count how many values contain a dash 
    dash_count = df[column_name].str.contains("-").sum()
    
    # Count how many values contain the letter 'm' 
    m_count = df[column_name].str.contains("m").sum()
    
    # Calculate and print the percentage of rows that contain any of these anomalies
    print(
        ((comma_count + dash_count + m_count) / len(df[column_name])) * 100
    )
    
    total_anomalies = (comma_count + dash_count + m_count)
    
    # Total number of rows in the column
    total_num_rows = len(df[column_name])
    
    numbers = [total_anomalies, total_num_rows - total_anomalies]
    labels = ["dirty values", "clean values"]
    
    # creating pie chart
    fig, ax = plt.subplots(figsize=(4, 4))  
    ax.pie(
        numbers,
        labels=labels,
        autopct='%1.1f%%',
        startangle=90,
        radius=0.7 
    )
    ax.set_title(column_name)
    ax.axis('equal')        
    plt.tight_layout()
    plt.show()


cleaning_col(column_name="Length_Overall_Ft")
cleaning_col(column_name="Beam_Ft")
cleaning_col(column_name="Draft_Ft")

In [ ]:
def cleaning_more_data(column_name):

    df[column_name] = df[column_name].astype("string")

    # normalize text, change separators and remove units/symbols
    df[column_name] = (
        df[column_name]
        .str.replace(",", ".")
        .str.replace("-", "")
        .str.replace("m", "")
        .str.replace("feet", "")
    )
    
    first_dot = df[column_name].str.find(".")
    
    # remove leading/trailing whitespace
    df[column_name] = df[column_name].apply(lambda x: x.strip())

    df[column_name] = df[column_name].astype("float64")

    print(df[column_name])


# clean specific columns
cleaning_more_data(column_name="Length_Overall_Ft")
cleaning_more_data(column_name="Beam_Ft")
cleaning_more_data(column_name="Draft_Ft")

In [ ]:
df.head(10)

In [ ]:
thevals = df["Engine_Count"].astype("string")
counting = thevals.value_counts()

numba = counting.values
labels = counting.index.tolist()

# bar chart
fig, ax = plt.subplots(figsize=(6, 4))  

ax.bar(labels, numba)
ax.set_xlabel("Engine Count")
ax.set_ylabel("Frequency")
ax.set_title("Engine Count Distribution")

# make x labels readable
plt.xticks(rotation=45)

plt.tight_layout()
plt.show()

print(counting)


In [ ]:
# remove rows where Engine_Count equals '0'
df = df[df["Engine_Count"] != '0'] 

# remove rows where Engine_Count equals 'many'
df = df[df["Engine_Count"] != 'many'] 

# show Engine_Count column
df["Engine_Count"]

# convert Engine_Count to integer type
df["Engine_Count"] = df["Engine_Count"].astype("int")

In [ ]:
def unique_and_upper(column_name):
    
    df[column_name].unique()

    df[column_name] = df[column_name].astype("str").str.upper()
    
    print(df[column_name])

unique_and_upper(column_name="Engine_Type")
unique_and_upper(column_name="Fuel_Type")
unique_and_upper(column_name="Propulsion")

In [ ]:
df.head(10)

In [ ]:
# Normalize text columns
df['Propulsion'] = df['Propulsion'].astype(str).str.strip().str.lower()
df['Rig_Type']   = df['Rig_Type'].astype(str).str.strip().str.lower()

df['Rig_Type'] = df['Rig_Type'].replace({'': np.nan, 'na': np.nan, 'n/a': np.nan})

df['Mast_Count'] = pd.to_numeric(df['Mast_Count'], errors='coerce')

mask_power = df['Propulsion'] == 'power'
df.loc[mask_power, 'Mast_Count'] = 0

In [ ]:
# Only sailing boats
mask_sail = df['Propulsion'] == 'sail'

# Mask of sail rows with missing mast count
mask_missing_mast_sail = mask_sail & df['Mast_Count'].isna()

rig_to_mast = {
    'sloop': 1,
    'cat-rig': 1,
    'cutter': 1,
    'ketch': 2,
    'schooner': 2,
}

for rig, mast_val in rig_to_mast.items():
    m = mask_missing_mast_sail & (df['Rig_Type'] == rig)
    df.loc[m, 'Mast_Count'] = mast_val

In [ ]:
mask_still_missing_sail = mask_sail & df['Mast_Count'].isna()

df.loc[mask_still_missing_sail, 'Mast_Count'] = 1

In [ ]:
df['Propulsion'] = df['Propulsion'].astype(str).str.strip().str.lower()
df['Rig_Type']   = df['Rig_Type'].astype(str).str.strip().str.lower()

# Ensure Mast_Count is numeric
df['Mast_Count'] = pd.to_numeric(df['Mast_Count'], errors='coerce')


mask_power = df['Propulsion'] == 'power'
mask_missing_rig_power = mask_power & df['Rig_Type'].isna()

df.loc[mask_missing_rig_power, 'Rig_Type'] = 'none'


# impute by mast count
mask_sail = df['Propulsion'] == 'sail'
mask_missing_rig_sail = mask_sail & df['Rig_Type'].isna()

# Sailboats with 2+ masts -> ketch
multi_mast = mask_missing_rig_sail & (df['Mast_Count'] >= 2)
df.loc[multi_mast, 'Rig_Type'] = 'ketch'

# Sailboats with 1 or 0 masts -> sloop
single_mast = mask_missing_rig_sail & (df['Mast_Count'] < 2)
df.loc[single_mast, 'Rig_Type'] = 'sloop'


# safe fallback
mask_still_missing = df['Rig_Type'].isna()
df.loc[mask_still_missing, 'Rig_Type'] = 'unknown'


In [ ]:
df['Rig_Type'] = df['Rig_Type'].astype(str).str.strip().str.lower()

df['Rig_Type'] = df['Rig_Type'].replace({'': np.nan, 'nan': np.nan})

In [ ]:
mask_power = df['Propulsion'].str.lower() == 'power'

df.loc[mask_power, 'Rig_Type'] = 'none'

In [ ]:
df['Mast_Count'] = pd.to_numeric(df['Mast_Count'], errors='ignore')

mask_sail = df['Propulsion'].str.lower() == 'sail'
mask_missing = df['Rig_Type'].isna() & mask_sail

df.loc[mask_missing & (df['Mast_Count'] >= 2), 'Rig_Type'] = 'ketch'

df.loc[mask_missing & (df['Mast_Count'] < 2), 'Rig_Type'] = 'sloop'

df['Rig_Type'] = df['Rig_Type'].fillna('unknown')

Feature Store

In [ ]:
# writing pandas dataframe to session
# creating/overwriting a physical table,

session.write_pandas(
    df=df,
    table_name = "BOATS_SN_TABLE",
    database = "SNOW_ML_DB",
    schema = "SNOW_ML_SCHEMA",
    auto_create_table = True,
    overwrite = True
)

In [ ]:
boats_table = session.table("BOATS_SN_TABLE")

In [ ]:
from snowflake.snowpark.context import get_active_session


session.query_tag = {"origin":"sf_sit-is", "name":"ml_pipeline_fs", "version":{"major":1, "minor":0}}

print(f"role: {session.get_current_role()} | WH: {session.get_current_warehouse()} | DB.SCHEMA: {session.get_fully_qualified_current_schema()}")

In [ ]:
from snowflake.ml.feature_store import FeatureStore, CreationMode, FeatureView

# create feature store
fs = FeatureStore(
    session = session,
    database = "SNOW_ML_DB",
    name = "FS_SCHEMA",
    default_warehouse = "SNOW_ML_PIPELINE",
    creation_mode = CreationMode.CREATE_IF_NOT_EXIST
)

In [ ]:
from snowflake.ml.feature_store import Entity

# An entity is an abstraction over a set of primary keys used for 
# looking up feature data. An Entity represents a real-world "thing" 
# that has data associated with it.

boat_id_entity = Entity(
    name = "VESSEL_ID",
    join_keys = ["VESSEL_ID"],
    desc = "Unique boat id"
)

fs.register_entity(boat_id_entity)

boat_entity = Entity(
    name="BOAT_MODEL", 
    join_keys=["BOAT_MODEL"], 
    desc="Boat model"
)

fs.register_entity(boat_entity)

fs.list_entities().show()

Feature View

In [ ]:
from snowflake.ml.feature_store import FeatureView, Entity


# A feature view is a group of logically-related features that are 
# refreshed on the same schedule.

boats_premium_features = session.sql(
    """
        SELECT
        "Vessel_Id"                             AS VESSEL_ID,
        "Feature_Premium_Cruising_Score"        AS FEATURE_PREMIUM_CRUISING_SCORE,
    
        -- 0–100 → 0–1
        "Feature_Premium_Cruising_Score" / 100.0 AS PREMIUM_SCORE_NORM,
    
        -- Buckets
        CASE
            WHEN "Feature_Premium_Cruising_Score" < 40 THEN 'LOW'
            WHEN "Feature_Premium_Cruising_Score" < 60 THEN 'MEDIUM'
            WHEN "Feature_Premium_Cruising_Score" < 75 THEN 'HIGH'
            ELSE 'PREMIUM'
        END AS PREMIUM_SEGMENT
    
        FROM SNOW_ML_DB.SNOW_ML_SCHEMA.BOATS_SN_TABLE;

    """
)


boat_core_fv = FeatureView(
    name="boat_hull_fv_v2",
    entities=[boat_id_entity],
    feature_df=boats_premium_features,
    refresh_freq="1d",
    desc="""
          Premium cruising features per vessel, including the raw 0–100 
          premium cruising score, a 0–1 normalized version, a LOW/MEDIUM/HIGH/PREMIUM 
          segment label.
         """
)


boat_hull_fv_2 = fs.register_feature_view(
                    boat_core_fv, 
                    version = "1",
                    overwrite=True
                )

fs.list_feature_views().to_pandas()

In [ ]:
# Get registered feature view 
boat_fv = fs.get_feature_view("boat_hull_fv_v2", "1")

spine_df = session.sql(
    """
    SELECT
        "Target_Type"                   AS TARGET_TYPE,
        "Vessel_Id"                     AS VESSEL_ID,
        "Hull_Count"                    AS HULL_COUNT,
        "Hull_Type"                     AS HULL_TYPE,
        "Beam_To_Length_Ratio"          AS BEAM_TO_LENGTH_RATIO,
        "Length_Overall_Ft"             AS LENGTH_OVERALL_FT,
        "Beam_Ft"                       AS BEAM_FT,
        "Draft_Ft"                      AS DRAFT_FT,
        "Engine_Count"                  AS ENGINE_COUNT,
        "Engine_Type"                   AS ENGINE_TYPE,
        "Fuel_Type"                     AS FUEL_TYPE,
        "Propulsion"                    AS PROPULSION,
        "Mast_Count"                    AS MAST_COUNT,
        "Rig_Type"                      AS RIG_TYPE,
        "Year_Built"                    AS YEAR_BUILT
    FROM SNOW_ML_DB.SNOW_ML_SCHEMA.BOATS_SN_TABLE
    """
)

# Join the spine and features from feature view into one training set
training_sp = fs.generate_training_set(
    spine_df=spine_df,
    features=[boat_fv],
    spine_label_cols=["TARGET_TYPE"],  
)

# Bring into pandas for sklearn
df = training_sp.to_pandas()
print("Columns:", df.columns.tolist())

Model Training and Experiment Tracking

In [ ]:
from snowflake.snowpark.context import get_active_session
from snowflake.ml.experiment import ExperimentTracking  
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline as SkPipeline
from sklearn.metrics import classification_report, accuracy_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer


MAX_ROWS = 50000
if len(df) > MAX_ROWS:
    df = df.sample(n=MAX_ROWS, random_state=42)

# Filter to the 3 target classes
target_col = "TARGET_TYPE"
wanted_classes = ["MONOHULL", "CATAMARAN", "POWER CATAMARAN"]

df = df[df[target_col].isin(wanted_classes)].copy()

# Drop rows with missing label
df = df.dropna(subset=[target_col])

# Drop any class with fewer than 2 samples (safety for stratify=y)
counts = df[target_col].value_counts()
valid_classes = counts[counts >= 2].index
df = df[df[target_col].isin(valid_classes)].copy()


# Balance classes by downsampling to the smallest class size
min_count = df[target_col].value_counts().min()

df_balanced = (
    df.groupby(target_col, group_keys=False)
      .apply(lambda x: x.sample(n=min_count, random_state=42))
)

print(df_balanced[target_col].value_counts())


# Split features / label
label_column_name = target_col
id_cols = ["VESSEL_ID"]

X = df_balanced.drop(columns=[label_column_name] + id_cols)
y = df_balanced[label_column_name]


# Identify numeric vs categorical features
numeric_cols = X.select_dtypes(include=["number"]).columns.tolist()
categorical_cols = X.select_dtypes(exclude=["number"]).columns.tolist()


# Preprocessing pipelines
numeric_transformer = SkPipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
    ]
)

categorical_transformer = SkPipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_cols),
        ("cat", categorical_transformer, categorical_cols),
    ]
)


# Train / test split 
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,  
)

In [ ]:
# Snowflake ML Experiment with 3 runs
exp = ExperimentTracking(session=session)

# Create or select the experiment to log into
exp.set_experiment("BOATS_HULL_CLASSIFIER_EXP")

# Three different RF configs to compare
rf_configs = [
    {"run_name": "run_1_a",  "n_estimators": 50,  "max_depth": 12},
    {"run_name": "run_2_a",  "n_estimators": 100, "max_depth": 16},
    {"run_name": "run_3_a",  "n_estimators": 200, "max_depth": None},
]

for cfg in rf_configs:
    with exp.start_run(cfg["run_name"]):
        print(f"\n=== Starting run: {cfg['run_name']} ===")

        # Define model for this run
        clf = RandomForestClassifier(
            n_estimators=cfg["n_estimators"],
            max_depth=cfg["max_depth"],
            n_jobs=-1,
            random_state=42,
            class_weight="balanced",
        )

        model = SkPipeline(
            steps=[
                ("preprocess", preprocessor),
                ("clf", clf),
            ]
        )

        # Fit model
        print("Fitting model...")
        model.fit(X_train, y_train)

        # Evaluate model
        y_pred = model.predict(X_test)
        acc = accuracy_score(y_test, y_pred)

        print("\nAccuracy:", acc)
        print("\nClassification report:")
        print(classification_report(y_test, y_pred))

        # Log metrics & params using the ExperimentTracking API
        exp.log_metric("accuracy", acc)
        exp.log_param("n_estimators", cfg["n_estimators"])
        exp.log_param("max_depth", cfg["max_depth"])
        exp.log_param("class_weight", "balanced")

        print(f"=== Finished run: {cfg['run_name']} ===\n")

Model Registry

In [ ]:
from snowflake.ml.registry import Registry

# create model registry
reg = Registry( 
                session=session,    
                database_name = "SNOW_ML_DB",
                schema_name = "SNOW_ML_SCHEMA"
              )   

In [ ]:
from snowflake.ml.model import task, type_hints
 
mv = reg.log_model(
    model=model,                       
    model_name="BOAT_CLASSIFIER",    
    comment="RF hull-type classifier for MONOHULL, CATAMARAN and POWER CATAMARAN (using container runtime)",
    sample_input_data=X_train,         
    pip_requirements=[
        "scikit-learn",   
        "pandas",         
    ],
)

In [ ]:
LIST 'snow://model/BOAT_CLASSIFIER/versions/TRICKY_CAT_4';

In [ ]:
models = reg.show_models()
print(models)

In [ ]:
my_model = reg.get_model("BOAT_CLASSIFIER").version("TRICKY_CAT_CONTAINER")

my_model = my_model.load()
print(type(my_model))

In [ ]:
# # 1) Load table to pandas
# df_scoring = session.table("SNOW_ML_DB.SNOW_ML_SCHEMA.BOATS_SN_TABLE").to_pandas()

# # 2) Strip the double quotes from column names: '"Hull_Count"' -> 'Hull_Count'
# df_scoring.columns = [c.strip('"') for c in df_scoring.columns]

# print("Columns available for scoring:", df_scoring.columns.tolist())

# # 3) Build X_scoring with the exact same feature columns as training
# X_scoring = df_scoring[feature_cols].copy()

# print("X_scoring shape:", X_scoring.shape)
# print("X_scoring dtypes:\n", X_scoring.dtypes)


In [ ]:
feature_cols = list(X_train.columns)

print("Feature columns used in training:", feature_cols)

In [ ]:
# Full table
inference_sp_df = session.table("SNOW_ML_DB.SNOW_ML_SCHEMA.BOATS_SN_TABLE")
print(inference_sp_df)

feature_sp_df = inference_sp_df.drop('"Target_Type"')  

In [ ]:
# Load model version from the registry
reg = Registry(
    session=session,
    database_name="SNOW_ML_DB",
    schema_name="SNOW_ML_SCHEMA",
)

mv = reg.get_model("BOAT_CLASSIFIER").version("TRICKY_CAT_CONTAINER")
model = mv.load()

# Load source data from Snowflake into pandas
df_scoring = session.table("SNOW_ML_DB.SNOW_ML_SCHEMA.BOATS_SN_TABLE").to_pandas()

# Strip quotes: '"Hull_Count"' -> 'Hull_Count'
df_scoring.columns = [c.strip('"') for c in df_scoring.columns]

# Build X_scoring with the EXACT column names the pipeline expects (UPPERCASE)
X_scoring = pd.DataFrame(
    {
        "HULL_COUNT": df_scoring["Hull_Count"],
        "HULL_TYPE": df_scoring["Hull_Type"],
        "BEAM_TO_LENGTH_RATIO": df_scoring["Beam_To_Length_Ratio"],
        "LENGTH_OVERALL_FT": df_scoring["Length_Overall_Ft"],
        "BEAM_FT": df_scoring["Beam_Ft"],
        "DRAFT_FT": df_scoring["Draft_Ft"],
        "ENGINE_COUNT": df_scoring["Engine_Count"],
        "ENGINE_TYPE": df_scoring["Engine_Type"],
        "FUEL_TYPE": df_scoring["Fuel_Type"],
        "PROPULSION": df_scoring["Propulsion"],
        "MAST_COUNT": df_scoring["Mast_Count"],
        "RIG_TYPE": df_scoring["Rig_Type"],
        "YEAR_BUILT": df_scoring["Year_Built"],
        "FEATURE_PREMIUM_CRUISING_SCORE": df_scoring["Feature_Premium_Cruising_Score"],
    }
)

# Predict depending on model type
if isinstance(model, Pipeline):
    y_pred_raw = model.predict(X_scoring)

elif isinstance(model, xgb.Booster):
    
    dtest = xgb.DMatrix(
        X_scoring,
        feature_names=list(X_scoring.columns),
        enable_categorical=True,
    )
    y_pred_raw = model.predict(dtest)
    if y_pred_raw.ndim == 2:
        y_pred_raw = np.argmax(y_pred_raw, axis=1)
    else:
        y_pred_raw = (y_pred_raw > 0.5).astype(int)
else:
    y_pred_raw = model.predict(X_scoring)


# Compute overall accuracy before printing individual predictions
y_true = df_scoring["Target_Type"]

overall_accuracy = accuracy_score(y_true, y_pred_raw)
print(f"Overall accuracy on BOATS_SN_TABLE: {overall_accuracy:.4f}")


# Attach predictions back to the original df_scoring
df_scoring["PREDICTED_CLASS_INDEX"] = y_pred_raw


print(df_scoring[["Vessel_Id", "Target_Type", "PREDICTED_CLASS_INDEX"]].head(50))

In [ ]:
import pandas as pd
from snowflake.snowpark.context import get_active_session

# Write it to Snowflake as BOAT_HULL_SCORING 

# df_scoring should already exist from your scoring step
# and have at least: Target_Type, Vessel_Id, + features, y_pred_raw etc.

# 1) Make sure we have a clean prediction column name
if "PREDICTED_HULL_TYPE" not in df_scoring.columns:
    if "PREDICTED_CLASS_INDEX" in df_scoring.columns:
        df_scoring.rename(
            columns={"PREDICTED_CLASS_INDEX": "PREDICTED_HULL_TYPE"},
            inplace=True,
        )
    else:
        # If no prediction column yet, create it from y_pred_raw
        df_scoring["PREDICTED_HULL_TYPE"] = y_pred_raw

# 2) Add a synthetic timestamp for monitoring
df_scoring["SCORING_TS"] = pd.Timestamp.utcnow()

# 3) Build the monitoring DataFrame
monitor_df = df_scoring[[
    "SCORING_TS",          # timestamp for monitor
    "Vessel_Id",           # optional identifier
    "Target_Type",         # actual
    "PREDICTED_HULL_TYPE"  # prediction
]]

session.write_pandas(
    df=monitor_df,
    table_name="BOAT_HULL_SCORING",
    database="SNOW_ML_DB",
    schema="SNOW_ML_SCHEMA",
    auto_create_table=True,
    overwrite=True,        # overwrite for now; for prod you might append
)

print("Wrote BOAT_HULL_SCORING with shape:", monitor_df.shape)